# UC-04 Tutorial: Ambiguous Shared-Context + Review Queue

This tutorial runs `run_use_case_04_shared_context_ambiguous_review.py`.

Goal:
- Trigger `created` and `ambiguous` statuses
- Populate pending review queue
- Resolve manually from UI (recommended) or programmatically

Prerequisites:
- Open this notebook from inside the SEGB repository.
- Start backend API at `http://localhost:5000`.
- Python env:
  - `./.venv/bin/python -m pip install -e packages/semantic_log_generator`
  - `./.venv/bin/python -m pip install pydantic`
- Run the backend preflight cell below before executing the main use-case cell.
- UI routes can be opened on:
  - Production: `http://localhost:8080`
  - Development: `http://localhost:5173`


In [ ]:
# Ensure repository root is available in sys.path
import sys
from pathlib import Path

repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'examples').exists() and (candidate / 'packages' / 'semantic_log_generator').exists():
        repo_root = candidate
        break

if repo_root is None:
    raise RuntimeError('Repository root not found. Open this notebook inside the SEGB repository.')

extra_paths = [
    repo_root,
    repo_root / 'packages' / 'semantic_log_generator' / 'src',
    repo_root / 'apps' / 'backend' / 'src',
]
for path in extra_paths:
    path_str = str(path)
    if path.exists() and path_str not in sys.path:
        sys.path.insert(0, path_str)

print(f'Using repo root: {repo_root}')


In [ ]:
# Backend preflight (health checks)
import json
import urllib.error
import urllib.request

API_URL = 'http://localhost:5000'
REQUIRE_BACKEND = True


def _get_json(url: str) -> dict:
    with urllib.request.urlopen(url, timeout=5) as response:
        payload = response.read().decode('utf-8')
    return json.loads(payload)


def check_backend_health(api_url: str, *, require: bool) -> None:
    base = api_url.rstrip('/')
    live_url = f"{base}/healthz/live"
    ready_url = f"{base}/healthz/ready"

    try:
        live = _get_json(live_url)
        ready = _get_json(ready_url)
    except (urllib.error.URLError, TimeoutError, json.JSONDecodeError) as exc:
        message = f"Backend not reachable at {api_url}: {exc}"
        if require:
            raise RuntimeError(message) from exc
        print('WARNING:', message)
        return

    live_ok = bool(live.get('live'))
    ready_ok = bool(ready.get('ready')) and bool(ready.get('virtuoso'))

    print('healthz/live:', live)
    print('healthz/ready:', ready)

    if not (live_ok and ready_ok):
        message = f"Backend unhealthy at {api_url} (live_ok={live_ok}, ready_ok={ready_ok})"
        if require:
            raise RuntimeError(message)
        print('WARNING:', message)
        return

    print(f"Backend preflight OK -> {api_url}")


check_backend_health(API_URL, require=REQUIRE_BACKEND)


In [ ]:
from examples.simulations.http_helpers import ApiConfig
from examples.simulations.run_use_case_04_shared_context_ambiguous_review import run_shared_context_ambiguous_review_use_case

API_URL = 'http://localhost:5000'
TOKEN = None  # set JWT string if auth is enabled
DECISION = 'none'  # 'none' keeps case pending for manual UI review, 'accept' or 'reject' decides programmatically

api_config = ApiConfig(base_url=API_URL, token=TOKEN, timeout_seconds=20.0, verify_tls=True)
result = run_shared_context_ambiguous_review_use_case(api_config=api_config, decision=DECISION)

print('Use case: UC-04 ambiguous review')
print('Active context URI:', result.active_context_uri)
print('Ambiguous context URI:', result.ambiguous_context_uri)
print('Created status:', result.created_resolution.status)
print('Ambiguous status:', result.ambiguous_resolution.status)
print('Pending before:', result.review_queue_before.pending_count)
print('Pending after:', result.review_queue_after.pending_count)
print('Manual decision:', None if result.review_decision is None else result.review_decision.status)


## UI Walkthrough (Important)

Use the same route on prod (`http://localhost:8080`) or dev (`http://localhost:5173`).

1. Open `/shared-context`.
2. In the pending queue, locate the new case generated by this notebook.
3. Inspect candidate contexts and scores.
4. Click **Accept** or **Reject** and verify queue updates.
5. Check `stats` section to see merged/ambiguous counters.

## CLI Alternative

```bash
./.venv/bin/python -m examples.simulations.run_use_case_04_shared_context_ambiguous_review \
  --publish-url http://localhost:5000 \
  --decision accept \
  --no-print-ttl
```
